# Zurich Airport data report

This notebook documents the cleaning and merging pipeline for the Zurich Airport flight assignment.

## Sources

- `schedule_airport.csv`: flight schedule and actual times for 2019-2020.
- `airports-extended.csv`: airport reference data. This file has no header row.
- `weather_zurich_2019-2020.csv`: daily Zurich weather observations for the same period.

The goal is to create one flight-level dataframe that can be used to analyse and predict delay.

In [6]:
from pathlib import Path

import pandas as pd
from IPython.display import display

from src.data_cleaner import AirportDataCleaner
from src.data_merger import DataMerger

DATA_DIR = Path("../data")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 40)

## 1. Load and clean the three datasets

### Schedule cleaning

The schedule cleaner:

- normalizes column names;
- converts `-`, empty values, and other configured markers to missing values;
- keeps mixed categorical fields such as `IX1` and `IX2` as text;
- parses `STD` as a date;
- parses `STA_STD_ltc` and `ATA_ATD_ltc` as times;
- does not remove rows by default.

The delay-code columns are not converted to numbers because they contain both numeric and letter-based codes.

In [7]:
schedule_cleaner = AirportDataCleaner.from_schedule_csv(
DATA_DIR / "schedule_airport.csv"
)
schedule_before = schedule_cleaner.get_df().copy()
schedule = schedule_cleaner.clean_schedule()

print(f"Schedule rows before cleaning: {len(schedule_before):,}")
print(f"Schedule rows after cleaning:  {len(schedule):,}")
print(f"Rows removed:                 {len(schedule_before) - len(schedule):,}")
display(schedule.head())

Schedule rows before cleaning: 323,461
Schedule rows after cleaning:  323,461
Rows removed:                 0


,std,flt,sta_std_ltc,ata_atd_ltc,lsv,tar,gat,dl1,ix1,dl2,ix2,act,rwy,rwc,identifier,org/des
0,2019-01-01,A3850,10:15:00,10:13:41,L,A13,A75,NaN,NaN,NaN,NaN,A321,14,Standard,01.01.2019 10:15A3850,LGAV
1,2019-01-01,A3851,11:00:00,11:01:06,S,A13,A75,NaN,NaN,NaN,NaN,A321,28,Standard,01.01.2019 11:00A3851,LGAV
2,2019-01-01,AA092,08:50:00,08:45:34,L,E35,E35,NaN,NaN,NaN,NaN,B763,34,South_C1,01.01.2019 08:50AA092,KPHL
3,2019-01-01,AA093,11:45:00,11:49:00,S,E35,E35,85,NaN,NaN,NaN,B763,16,Standard,01.01.2019 11:45AA093,KPHL
4,2019-01-01,AF1114,08:45:00,08:37:28,L,B36,B36,NaN,NaN,NaN,NaN,A319,34,South_C1,01.01.2019 08:45AF1114,LFPG


### Airport reference cleaning

The airport file is loaded with explicit column names because the source has no header row. The cleaner converts coordinates, altitude, and timezone to numeric values, trims text, and treats standalone `\\N` as missing.

Rows are not removed because stations, ports, and locations without an IATA or ICAO code may still be useful reference records. Invalid values are reported for review instead.

In [8]:
airport_cleaner = AirportDataCleaner.from_airports_csv(
DATA_DIR / "airports-extended.csv"
)
airports_before = airport_cleaner.get_df().copy()
airports = airport_cleaner.clean_airports()

print(f"Airport rows before cleaning: {len(airports_before):,}")
print(f"Airport rows after cleaning:  {len(airports):,}")
print(f"Rows removed:                {len(airports_before) - len(airports):,}")
display(airports.head())

Airport rows before cleaning: 10,668
Airport rows after cleaning:  10,668
Rows removed:                0


,airport_id,name,city,country,iata,icao,latitude,longitude,altitude,timezone,dst,tz_database_time_zone,type,source
0,1,Goroka Airport,Goroka,Papua New Guinea,GKA,AYGA,-6.081690,145.391998,5282,10.0,U,Pacific/Port_Moresby,airport,OurAirports
1,2,Madang Airport,Madang,Papua New Guinea,MAG,AYMD,-5.207080,145.789001,20,10.0,U,Pacific/Port_Moresby,airport,OurAirports
2,3,Mount Hagen Kagamuga Airport,Mount Hagen,Papua New Guinea,HGU,AYMH,-5.826790,144.296005,5388,10.0,U,Pacific/Port_Moresby,airport,OurAirports
3,4,Nadzab Airport,Nadzab,Papua New Guinea,LAE,AYNZ,-6.569803,146.725977,239,10.0,U,Pacific/Port_Moresby,airport,OurAirports
4,5,Port Moresby Jacksons International ...,Port Moresby,Papua New Guinea,POM,AYPY,-9.443380,147.220001,146,10.0,U,Pacific/Port_Moresby,airport,OurAirports


### Weather cleaning

The new weather file contains one observation per day. The cleaner therefore parses `date`, converts the weather measurements to numeric values, and checks daily continuity. Missing snow or wind observations remain missing; they are not changed to zero because missing and zero have different meanings.

In [9]:
weather_cleaner = AirportDataCleaner.from_weather_csv(
DATA_DIR / "weather_zurich_2019-2020.csv"
)
weather_before = weather_cleaner.get_df().copy()
weather = weather_cleaner.clean_weather()

print(f"Weather rows before cleaning: {len(weather_before):,}")
print(f"Weather rows after cleaning:  {len(weather):,}")
print(f"Rows removed:                {len(weather_before) - len(weather):,}")
display(weather.head())

Weather rows before cleaning: 731
Weather rows after cleaning:  731
Rows removed:                0


,date,tavg,tmin,tmax,prcp,snow,wdir,wspd,wpgt,pres,tsun
0,2019-01-01 00:00:00,4.5,4.0,5.0,0.0,NaN,NaN,8.6,30.0,1033.3,NaN
1,2019-01-02 00:00:00,1.9,-4.3,4.4,2.0,NaN,NaN,12.6,43.0,1034.6,NaN
2,2019-01-03 00:00:00,-3.1,-7.4,0.3,0.0,NaN,NaN,3.4,14.8,1039.6,NaN
3,2019-01-04 00:00:00,-0.5,-4.1,2.2,0.0,NaN,NaN,5.3,14.8,1038.1,NaN
4,2019-01-05 00:00:00,0.5,-0.3,1.7,3.6,NaN,NaN,7.9,32.0,1034.7,NaN


## 2. Quality report

The quality report distinguishes missing values from invalid values. Validation does not silently delete invalid rows. This keeps the cleaning decisions auditable.

In [10]:
def show_quality_summary(name, cleaner):
    report = cleaner.quality_report()
    print(f"{name}")
    print(f"Rows: {report['row_count']:,} | Columns: {report['column_count']}")
    print(f"Complete duplicate rows: {report['duplicate_rows']:,}")
    missing = report["missing_values"]
    print("Missing values:")
    display(missing[missing > 0].rename("missing_count").to_frame())

show_quality_summary("Schedule", schedule_cleaner)
show_quality_summary("Airports", airport_cleaner)
show_quality_summary("Weather", weather_cleaner)

Schedule
Rows: 323,461 | Columns: 16
Complete duplicate rows: 0
Missing values:


,missing_count
tar,418
gat,398
dl1,143464
ix1,281704
dl2,247118
ix2,290779
rwc,2323
org/des,94


Airports
Rows: 10,668 | Columns: 14
Complete duplicate rows: 0
Missing values:


,missing_count
city,47
iata,4202
icao,3044
timezone,314
dst,314
tz_database_time_zone,2282


Weather
Rows: 731 | Columns: 11
Complete duplicate rows: 0
Missing values:


,missing_count
prcp,24
snow,731
wdir,731
tsun,731


In [11]:
def validation_counts(cleaner, report_key):
    validation = cleaner.quality_report().get(report_key, {})
    return pd.Series({key: len(value) for key, value in validation.items()})

print("Airport validation counts")
display(validation_counts(airport_cleaner, "airport_validation"))

print("Weather validation counts")
display(validation_counts(weather_cleaner, "weather_validation"))

Airport validation counts


invalid_latitude       0
invalid_longitude      0
invalid_timezone       0
invalid_airport_id     0
invalid_type           0
invalid_source         0
invalid_iata           2
invalid_icao          18
dtype: int64

Weather validation counts


invalid_wdir    0
invalid_wspd    0
invalid_wpgt    0
invalid_pres    0
invalid_tsun    0
dtype: int64

## 3. Create the delay target

`STD` and `STA_STD_ltc` describe the planned flight datetime. `ATA_ATD_ltc` describes the actual datetime. The merger calculates:

`delay_minutes = actual_datetime - scheduled_datetime`

A flight is classified as delayed when the delay exceeds 15 minutes. The original date/time columns are retained until the final modelling step so the calculation remains traceable.

In [12]:
merger = DataMerger(schedule)
merger.create_delay_columns(delayed_after_minutes=15)

print("Delay summary in minutes")
display(merger.get_df()["delay_minutes"].describe())
print("Delayed flights:")
print(merger.get_df()["is_delayed"].value_counts(dropna=False))
display(
    merger.get_df()[
        ["scheduled_datetime", "actual_datetime", "delay_minutes", "is_delayed"]
    ].head()
)

Delay summary in minutes


count    323461.000000
mean          8.281126
std          23.735319
min         -97.650000
25%          -3.000000
50%           3.316667
75%          14.000000
max         984.000000
Name: delay_minutes, dtype: float64

Delayed flights:
is_delayed
False    248695
True      74766
Name: count, dtype: int64


,scheduled_datetime,actual_datetime,delay_minutes,is_delayed
0,2019-01-01 10:15:00,2019-01-01 10:13:41,-1.316667,False
1,2019-01-01 11:00:00,2019-01-01 11:01:06,1.100000,False
2,2019-01-01 08:50:00,2019-01-01 08:45:34,-4.433333,False
3,2019-01-01 11:45:00,2019-01-01 11:49:00,4.000000,False
4,2019-01-01 08:45:00,2019-01-01 08:37:28,-7.533333,False


## 4. Merge airport metadata

The schedule's `Org/Des` value is an ICAO airport code. It is matched to `airports.icao` with a left join. A left join is intentional: every flight remains in the analysis, even when its airport code is not present in the reference table.

Only useful descriptive and geographic airport fields are added. The airport ID, data source, DST flag, and timezone database name are not needed for the first delay graphs.

In [13]:
rows_before_airport_merge = len(merger.get_df())
merger.merge_airports(airports)
merged_after_airports = merger.get_df()

airport_fields = [
    "name_airport", "city_airport", "country_airport",
    "latitude", "longitude", "altitude", "type",
]
available_airport_fields = [
    column for column in airport_fields if column in merged_after_airports
]
print(f"Rows before airport merge: {rows_before_airport_merge:,}")
print(f"Rows after airport merge:  {len(merged_after_airports):,}")
print("Unmatched airport rows:")
display(merged_after_airports[available_airport_fields].isna().all(axis=1).value_counts())
display(merged_after_airports.head())

Rows before airport merge: 323,461
Rows after airport merge:  323,461
Unmatched airport rows:


False    323203
True        258
Name: count, dtype: int64

,std,flt,sta_std_ltc,ata_atd_ltc,lsv,tar,gat,dl1,ix1,dl2,ix2,act,rwy,rwc,identifier,org/des,scheduled_datetime,actual_datetime,delay_minutes,is_delayed,iata,name,city,country,latitude,longitude,altitude,type
0,2019-01-01,A3850,10:15:00,10:13:41,L,A13,A75,NaN,NaN,NaN,NaN,A321,14,Standard,01.01.2019 10:15A3850,LGAV,2019-01-01 10:15:00,2019-01-01 10:13:41,-1.316667,False,ATH,Eleftherios Venizelos International ...,Athens,Greece,37.936401,23.944500,308.0,airport
1,2019-01-01,A3851,11:00:00,11:01:06,S,A13,A75,NaN,NaN,NaN,NaN,A321,28,Standard,01.01.2019 11:00A3851,LGAV,2019-01-01 11:00:00,2019-01-01 11:01:06,1.100000,False,ATH,Eleftherios Venizelos International ...,Athens,Greece,37.936401,23.944500,308.0,airport
2,2019-01-01,AA092,08:50:00,08:45:34,L,E35,E35,NaN,NaN,NaN,NaN,B763,34,South_C1,01.01.2019 08:50AA092,KPHL,2019-01-01 08:50:00,2019-01-01 08:45:34,-4.433333,False,PHL,Philadelphia International Airport,Philadelphia,United States,39.871899,-75.241096,36.0,airport
3,2019-01-01,AA093,11:45:00,11:49:00,S,E35,E35,85,NaN,NaN,NaN,B763,16,Standard,01.01.2019 11:45AA093,KPHL,2019-01-01 11:45:00,2019-01-01 11:49:00,4.000000,False,PHL,Philadelphia International Airport,Philadelphia,United States,39.871899,-75.241096,36.0,airport
4,2019-01-01,AF1114,08:45:00,08:37:28,L,B36,B36,NaN,NaN,NaN,NaN,A319,34,South_C1,01.01.2019 08:45AF1114,LFPG,2019-01-01 08:45:00,2019-01-01 08:37:28,-7.533333,False,CDG,Charles de Gaulle International Airport,Paris,France,49.012798,2.550000,392.0,airport


## 5. Merge daily weather data

The schedule has a flight-level datetime, while the weather file has one row per date at midnight. The merger uses the nearest weather date with a 12-hour tolerance. This attaches the weather observation for the same calendar day to each flight without requiring an exact timestamp match.

The weather columns are suffixed with `_weather` if a name conflicts with an existing schedule column.

In [14]:
rows_before_weather_merge = len(merger.get_df())
merger.merge_weather(weather)
merged = merger.get_df()

print(f"Rows before weather merge: {rows_before_weather_merge:,}")
print(f"Rows after weather merge:  {len(merged):,}")
weather_columns = [
    column for column in weather.columns if column != "date"
]
weather_columns_present = [
    column for column in weather_columns if column in merged.columns
]
print(f"Flights with matched average temperature: {merged['tavg'].notna().sum():,}")
print(f"Flights without matched weather: {merged['tavg'].isna().sum():,}")
display(merged[["scheduled_datetime", "date", "tavg", "tmin", "tmax", "delay_minutes"]].head())

Rows before weather merge: 323,461
Rows after weather merge:  323,461
Flights with matched average temperature: 323,418
Flights without matched weather: 43


,scheduled_datetime,date,tavg,tmin,tmax,delay_minutes
0,2019-01-01 06:05:00,2019-01-01,4.5,4.0,5.0,-2.000000
1,2019-01-01 06:10:00,2019-01-01,4.5,4.0,5.0,6.966667
2,2019-01-01 06:10:00,2019-01-01,4.5,4.0,5.0,5.450000
3,2019-01-01 06:10:00,2019-01-01,4.5,4.0,5.0,2.533333
4,2019-01-01 06:10:00,2019-01-01,4.5,4.0,5.0,0.366667


## 6. Final modelling dataframe

The final step removes information that is only known after the flight. Keeping those columns would create target leakage: the model would receive the answer, or a close description of the answer, as an input.

Removed from the modelling dataframe:

- `actual_datetime` and `ATA_ATD_ltc`: used to calculate the target;
- `DL1`, `DL2`, `IX1`, `IX2`: delay outcomes and delay categories;
- `GAT`: actual gate information;
- `Identifier`: redundant composite identifier.

The delay columns `delay_minutes` and `is_delayed` remain because they are the target variables.

In [15]:
rows_before_model_cleanup = len(merged)
model_data = merger.prepare_model_data()

print(f"Final modelling rows:    {len(model_data):,}")
print(f"Final modelling columns: {len(model_data.columns)}")
print(f"Rows removed:            {rows_before_model_cleanup - len(model_data):,}")
print("Final columns:")
print(list(model_data.columns))
display(model_data.head())

Final modelling rows:    323,461
Final modelling columns: 31
Rows removed:            0
Final columns:
['std', 'flt', 'sta_std_ltc', 'lsv', 'tar', 'act', 'rwy', 'rwc', 'org/des', 'scheduled_datetime', 'delay_minutes', 'is_delayed', 'iata', 'name', 'city', 'country', 'latitude', 'longitude', 'altitude', 'type', 'date', 'tavg', 'tmin', 'tmax', 'prcp', 'snow', 'wdir', 'wspd', 'wpgt', 'pres', 'tsun']


,std,flt,sta_std_ltc,lsv,tar,act,rwy,rwc,org/des,scheduled_datetime,delay_minutes,is_delayed,iata,name,city,country,latitude,longitude,altitude,type,date,tavg,tmin,tmax,prcp,snow,wdir,wspd,wpgt,pres,tsun
0,2019-01-01,WK214,06:05:00,S,A11,A320,32,South_C1,GCTS,2019-01-01 06:05:00,-2.000000,False,TFS,Tenerife South Airport,Tenerife,Spain,28.044500,-16.572500,209.0,airport,2019-01-01,4.5,4.0,5.0,0.0,NaN,NaN,8.6,30.0,1033.3,NaN
1,2019-01-01,LX289,06:10:00,L,B43,A343,34,South_C1,FAJS,2019-01-01 06:10:00,6.966667,False,JNB,OR Tambo International Airport,Johannesburg,South Africa,-26.139200,28.246000,5558.0,airport,2019-01-01,4.5,4.0,5.0,0.0,NaN,NaN,8.6,30.0,1033.3,NaN
2,2019-01-01,LX139,06:10:00,L,B39,B77W,34,South_C1,VHHH,2019-01-01 06:10:00,5.450000,False,HKG,Chek Lap Kok International Airport,Hong Kong,Hong Kong,22.308901,113.915001,28.0,airport,2019-01-01,4.5,4.0,5.0,0.0,NaN,NaN,8.6,30.0,1033.3,NaN
3,2019-01-01,LX155,06:10:00,L,E26,A333,34,South_C1,VABB,2019-01-01 06:10:00,2.533333,False,BOM,Chhatrapati Shivaji International Ai...,Mumbai,India,19.088699,72.867897,39.0,airport,2019-01-01,4.5,4.0,5.0,0.0,NaN,NaN,8.6,30.0,1033.3,NaN
4,2019-01-01,WK091,06:10:00,L,E20,A343,34,South_C1,SAEZ,2019-01-01 06:10:00,0.366667,False,EZE,Ministro Pistarini International Air...,Buenos Aires,Argentina,-34.822200,-58.535800,67.0,airport,2019-01-01,4.5,4.0,5.0,0.0,NaN,NaN,8.6,30.0,1033.3,NaN


## Result

The resulting `model_data` dataframe has one row per flight, a calculated delay target, airport metadata where the ICAO code matched, and weather variables from the matching Zurich date. It is ready for exploratory graphs such as delay by hour, aircraft type, destination, runway, season, temperature, precipitation, or wind conditions.